# 12장. 선호로 학습하기: DPO

정답 대신 '두 답 가운데 어느 쪽이 나은가'로 학습시킨다. 모델이 직접 쓴 영화평으로 선호 자료를 만들고, DPO 손실을 직접 계산해 11장의 LoRA로 학습시키며, β에 따라 무엇이 나아지고 무엇이 나빠지는지 잰다.

**준비:** `pip install "transformers<5" huggingface_hub`. 첫 셀이 Qwen3-0.6B(약 1.5GB)를 `data/models/`에 받는다. NSMC의 `ratings_train.txt`, `ratings_test.txt`도 필요하다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 46분쯤 걸렸다.

## 12.1 선호 자료 만들기

In [ ]:
import os, re, json, math, time, sys, random, platform, warnings, collections
import numpy as np
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = snapshot_download("Qwen/Qwen3-0.6B", local_dir="data/models/Qwen__Qwen3-0.6B",
                         allow_patterns=["*.json", "*.txt", "*.md", "LICENSE*", "*.safetensors"])
tok = AutoTokenizer.from_pretrained(PATH)
tok.padding_side = "left"                                         # 여러 질문을 묶어 만들 때는 왼쪽 패딩(6장)
END = tok.convert_tokens_to_ids("<|im_end|>")
load = lambda: AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float32)
print(sys.version.split()[0], torch.__version__, transformers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 판정기: NSMC 영화평 10만 개로 학습시킨 나이브 베이즈. 어절과 글자 두 개씩을 특징으로 쓴다.
def read(name):
    rows = [line.split("\t") for line in open(name, encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
    return [(text.strip(), int(label)) for _, text, label in rows if text.strip()]

train_all, test_all = read("ratings_train.txt"), read("ratings_test.txt")
feats = lambda s: set(s.split()) | {s[i:i + 2] for i in range(len(s) - 1)}
count, docs = [collections.Counter(), collections.Counter()], [0, 0]
for text, label in train_all[20000:120000]:
    count[label].update(feats(text)); docs[label] += 1
vocab = len(set(count[0]) | set(count[1]))
total = [sum(c.values()) for c in count]
def score(text):
    """판정 점수: 긍정일 로그 오즈(나트). 0보다 크면 긍정 쪽이다."""
    lp = [math.log(docs[y] / sum(docs)) + sum(math.log((count[y][f] + 1) / (total[y] + vocab)) for f in feats(text)) for y in (0, 1)]
    return lp[1] - lp[0]
test = test_all[:5000]
print(f"판정기: 학습 {sum(docs):,}개, 특징 {vocab:,}개, 시험 {len(test):,}개 정확도 {np.mean([(score(t) > 0) == y for t, y in test]):.1%}")
for t in ["배우들의 연기가 정말 좋았다", "시간 아까운 영화", "그냥 그랬다"]:
    print(f"  {t} → {score(t):+.2f}")

In [ ]:
# 질문 400개: NSMC 영화평의 첫 어절(한글 2~4글자)을 긍정·부정 영화평에서 반씩 뽑아 '이 낱말로 시작하는 영화평'을 쓰게 한다
def first_words(label, n):
    w = [t.split()[0] for t, y in train_all[:12000] if y == label and re.fullmatch(r"[가-힣]{2,4}", t.split()[0])]
    random.Random(label).shuffle(w)
    return list(dict.fromkeys(w))[:n]
WORDS = first_words(0, 200) + first_words(1, 200)
random.Random(0).shuffle(WORDS)
TRAIN_W, EVAL_W = WORDS[:300], WORDS[300:]
ASK = "'{}'(으)로 시작하는 짧은 영화평을 한 문장으로 쓰세요."
prompt_ids = lambda w: tok.apply_chat_template([{"role": "user", "content": ASK.format(w)}], tokenize=True,
                                               add_generation_prompt=True, enable_thinking=False)

def sample(model, words, n, seed=0, greedy=False, max_new=48):
    """질문마다 답 n개를 뽑는다(모델 카드의 비생각 모드 권장값: 온도 0.7, top-p 0.8, top-k 20). 답은 토큰 열로 돌려준다."""
    torch.manual_seed(seed); out = []
    model.eval()
    for k in range(0, len(words), 8):
        ps = [prompt_ids(w) for w in words[k:k + 8]]
        L = max(map(len, ps))
        ids = torch.tensor([[tok.pad_token_id] * (L - len(p)) + p for p in ps])
        mask = (torch.arange(L)[None] >= torch.tensor([L - len(p) for p in ps])[:, None]).long()
        with torch.inference_mode():
            g = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new, num_return_sequences=n, eos_token_id=[END, tok.eos_token_id],
                               pad_token_id=tok.pad_token_id, **({"do_sample": False, "temperature": None, "top_p": None, "top_k": None} if greedy else
                                                                  {"do_sample": True, "temperature": 0.7, "top_p": 0.8, "top_k": 20}))
        for i, row in enumerate(g[:, L:].tolist()):
            row = row[:row.index(END) + 1] if END in row else [t for t in row if t != tok.pad_token_id][:max_new]
            out.append((ps[i // n], row))
    return out

base = load()
lap = time.perf_counter()
pool = sample(base, WORDS, 4)
text = lambda r: tok.decode(r, skip_special_tokens=True).strip()
S = [score(text(r)) for _, r in pool]
print(f"답 {len(pool)}개를 {time.perf_counter() - lap:.0f}초에 만들었다. 판정 점수가 0보다 작은(부정 쪽) 답 {np.mean(np.array(S) < 0):.1%}, 평균 {len(''.join(text(r) for _, r in pool)) / len(pool):.1f}글자")
for (p, r), s in list(zip(pool, S))[:4]:
    print(f"  {s:+7.2f}  {text(r)[:60]}")

In [ ]:
# 질문마다 네 답 가운데 판정 점수가 가장 낮은 답을 좋은 답, 가장 높은 답을 나쁜 답으로 짝짓는다(점수 차 4 이상만)
def make_pairs(words, offset):
    pairs = []
    for k in range(len(words)):
        group = [(S[offset + 4 * k + j], pool[offset + 4 * k + j]) for j in range(4)]
        (s_good, (p, good)), (s_bad, (_, bad)) = min(group, key=lambda g: g[0]), max(group, key=lambda g: g[0])
        if s_bad - s_good >= 4 and good != bad:
            pairs.append({"prompt": p, "good": good, "bad": bad, "s_good": s_good, "s_bad": s_bad})
    return pairs
TRAIN_P, EVAL_P = make_pairs(TRAIN_W, 0), make_pairs(EVAL_W, 4 * len(TRAIN_W))
chars = lambda r: len(text(r))
print(f"선호 쌍: 학습 {len(TRAIN_P)}개(질문 {len(TRAIN_W)}개 가운데), 검증 {len(EVAL_P)}개")
print(f"좋은 답: 점수 평균 {np.mean([q['s_good'] for q in TRAIN_P]):+.2f}, 부정 쪽 {np.mean([q['s_good'] < 0 for q in TRAIN_P]):.1%}, 길이 평균 {np.mean([chars(q['good']) for q in TRAIN_P]):.1f}글자")
print(f"나쁜 답: 점수 평균 {np.mean([q['s_bad'] for q in TRAIN_P]):+.2f}, 부정 쪽 {np.mean([q['s_bad'] < 0 for q in TRAIN_P]):.1%}, 길이 평균 {np.mean([chars(q['bad']) for q in TRAIN_P]):.1f}글자")
print(f"좋은 답이 더 짧은 쌍 {np.mean([chars(q['good']) < chars(q['bad']) for q in TRAIN_P]):.1%}")
print(f"네 답 모두에서 판정 점수와 길이의 상관 {np.corrcoef(S, [chars(r) for _, r in pool])[0, 1]:.2f}")
q = TRAIN_P[0]
print("예) 질문:", tok.decode(q["prompt"]).split("user\n")[1].split("<|im_end|>")[0])
print(f"   좋은 답({q['s_good']:+.2f}): {text(q['good'])}")
print(f"   나쁜 답({q['s_bad']:+.2f}): {text(q['bad'])}")

## 12.2 DPO 손실 직접 계산하기

In [ ]:
def seq_logp(model, items):
    """(질문 토큰, 답 토큰) 묶음마다 답 토큰의 로그 확률 합 log π(답 | 질문)를 구한다. 오른쪽 패딩으로 묶는다."""
    L = max(len(p) + len(r) for p, r in items)
    ids = torch.tensor([p + r + [tok.pad_token_id] * (L - len(p) - len(r)) for p, r in items])
    mask = torch.tensor([[1] * (len(p) + len(r)) + [0] * (L - len(p) - len(r)) for p, r in items])
    on = torch.tensor([[0] * len(p) + [1] * len(r) + [0] * (L - len(p) - len(r)) for p, r in items])[:, 1:]
    logits = model(input_ids=ids, attention_mask=mask).logits[:, :-1]
    lp = torch.gather(F.log_softmax(logits, -1), 2, ids[:, 1:, None])[..., 0]
    return (lp * on).sum(1)

# 확인: Hugging Face 모델의 손실(답 토큰의 평균 음의 로그 확률) × 답 토큰 수와 같아야 한다
p, r = TRAIN_P[0]["prompt"], TRAIN_P[0]["good"]
with torch.inference_mode():
    mine = seq_logp(base, [(p, r)]).item()
    hf = base(input_ids=torch.tensor([p + r]), labels=torch.tensor([[-100] * len(p) + r])).loss.item() * len(r)
print(f"log π(좋은 답): 직접 {mine:.4f}, Hugging Face 손실로 {-hf:.4f}")

# 참조 모델(학습 전 모델)의 로그 확률은 학습 내내 바뀌지 않으므로 미리 한 번 구해 둔다
with torch.inference_mode():
    for P in (TRAIN_P, EVAL_P):
        for k in range(0, len(P), 8):
            batch = P[k:k + 8]
            lp = seq_logp(base, [(q["prompt"], q["good"]) for q in batch] + [(q["prompt"], q["bad"]) for q in batch])
            for i, q in enumerate(batch):
                q["ref_good"], q["ref_bad"] = lp[i].item(), lp[len(batch) + i].item()
print(f"참조 모델: log π(좋은 답) 평균 {np.mean([q['ref_good'] for q in TRAIN_P]):.2f}, log π(나쁜 답) 평균 {np.mean([q['ref_bad'] for q in TRAIN_P]):.2f}")

In [ ]:
def dpo_loss(lp_good, lp_bad, ref_good, ref_bad, beta):
    """DPO 손실 = -log σ(β · [(log π(좋은) - log π_ref(좋은)) - (log π(나쁜) - log π_ref(나쁜))])"""
    r_good, r_bad = beta * (lp_good - ref_good), beta * (lp_bad - ref_bad)    # 암묵적 보상
    return -F.logsigmoid(r_good - r_bad), r_good, r_bad

# 학습 전(정책 = 참조)에는 보상 차가 0이라 손실이 log 2여야 한다
q = TRAIN_P[0]
with torch.inference_mode():
    lp = seq_logp(base, [(q["prompt"], q["good"]), (q["prompt"], q["bad"])])
loss, rg, rb = dpo_loss(lp[0], lp[1], torch.tensor(q["ref_good"]), torch.tensor(q["ref_bad"]), 0.1)
print(f"학습 전 손실 {loss.item():.4f}, log 2 = {math.log(2):.4f}, 보상 {rg.item():+.4f} {rb.item():+.4f}")
# 정책이 좋은 답의 로그 확률을 1만큼 올리고 나쁜 답을 1만큼 내렸다면
for beta in [0.01, 0.1, 1.0]:
    l, _, _ = dpo_loss(torch.tensor(q["ref_good"] + 1), torch.tensor(q["ref_bad"] - 1), torch.tensor(q["ref_good"]), torch.tensor(q["ref_bad"]), beta)
    print(f"  β={beta}: 보상 차 {2 * beta:.2f}, 손실 {l.item():.4f}")

## 12.3 DPO로 학습시키기

In [ ]:
class LoRALinear(torch.nn.Module):
    """11장의 LoRA 층. on을 False로 두면 LoRA를 끄고 원래 층(참조 모델)으로 계산한다."""
    on = True
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = torch.nn.Parameter(torch.empty(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        return y + (x @ self.A.T) @ self.B.T * self.scale if LoRALinear.on else y

def lora_model(seed=0, r=8):
    torch.manual_seed(seed)
    model = load()
    for p in model.parameters():
        p.requires_grad_(False)
    for mod in list(model.modules()):
        for name, sub in list(mod.named_children()):
            if name in ("q_proj", "k_proj", "v_proj", "o_proj") and isinstance(sub, torch.nn.Linear):
                setattr(mod, name, LoRALinear(sub, r, 2 * r))
    return model

def reply(model, question, n=48):
    """채팅 템플릿에 넣은 질문에 탐욕 디코딩으로 답한다(9~11장과 같다)."""
    x = torch.tensor([tok.apply_chat_template([{"role": "user", "content": question}], tokenize=True, add_generation_prompt=True, enable_thinking=False)])
    with torch.inference_mode():
        g = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=n, do_sample=False, temperature=None, top_p=None, top_k=None,
                           eos_token_id=[END, tok.eos_token_id], pad_token_id=tok.pad_token_id)
    return tok.decode(g[0, x.shape[1]:], skip_special_tokens=True).strip()

QA = [("대한민국의 수도는 어디인가요?", ["서울"]), ("일본의 수도는 어디인가요?", ["도쿄"]), ("물의 화학식은 무엇인가요?", ["H2O", "H₂O"]),
      ("1년은 몇 개월인가요?", ["12개월", "12 개월", "12달"]), ("세종대왕이 만든 문자는 무엇인가요?", ["한글", "훈민정음"]),
      ("태양계에서 가장 큰 행성은 무엇인가요?", ["목성"]), ("지구에서 가장 넓은 바다는 무엇인가요?", ["태평양"]), ("7 곱하기 8은 얼마인가요?", ["56"]),
      ("100에서 37을 빼면 얼마인가요?", ["63"]), ("남한에서 가장 높은 산은 무엇인가요?", ["한라산"]), ("임진왜란이 일어난 해는 언제인가요?", ["1592"]),
      ("삼각형 세 내각의 합은 몇 도인가요?", ["180"]), ("사과를 영어로 하면 무엇인가요?", ["apple", "Apple"]),
      ("서울을 가로질러 흐르는 큰 강은 무엇인가요?", ["한강"]), ("대한민국 국기의 이름은 무엇인가요?", ["태극기"]),
      ("1킬로미터는 몇 미터인가요?", ["1000", "1,000"]), ("식물 세포에서 광합성을 하는 기관은 무엇인가요?", ["엽록체"]),
      ("컴퓨터에서 1바이트는 몇 비트인가요?", ["8비트", "8 비트", "8개"]), ("소설 '어린 왕자'를 쓴 작가는 누구인가요?", ["생텍쥐페리"]),
      ("빛의 삼원색은 빨강, 초록과 무엇인가요?", ["파랑", "파란"])]
OTHER = ["다음 영화평을 영어로 번역하세요.\n영화평: 배우들의 연기가 정말 좋았다", "다음 영화평을 한 문장으로 요약하세요.\n영화평: 처음에는 지루했지만 후반부 반전이 대단해서 끝까지 몰입했다",
         "'영화'로 시작하는 짧은 시를 한 줄 쓰세요.", "이 영화 어때요? 짧게 추천해 주세요."]
def evaluate(model, show=False):
    """검증 질문 100개에 두 답씩(시드 고정) 써서 판정 점수, 부정 쪽 비율, 길이, 다양성, 참조 모델과의 KL을 재고, 검증 선호 쌍과 상식 20문항도 잰다."""
    outs = sample(model, EVAL_W, 2, seed=1)
    sc = [score(text(r)) for _, r in outs]
    grams = [text(r)[i:i + 2] for _, r in outs for i in range(len(text(r)) - 1)]
    with torch.inference_mode():
        LoRALinear.on = True;  lp = torch.cat([seq_logp(model, outs[k:k + 16]) for k in range(0, len(outs), 16)])
        LoRALinear.on = False; ref = torch.cat([seq_logp(model, outs[k:k + 16]) for k in range(0, len(outs), 16)])
        LoRALinear.on = True
        pg = torch.cat([seq_logp(model, [(q["prompt"], q["good"]) for q in EVAL_P[k:k + 8]]) for k in range(0, len(EVAL_P), 8)])
        pb = torch.cat([seq_logp(model, [(q["prompt"], q["bad"]) for q in EVAL_P[k:k + 8]]) for k in range(0, len(EVAL_P), 8)])
    margin = (pg - torch.tensor([q["ref_good"] for q in EVAL_P])) - (pb - torch.tensor([q["ref_bad"] for q in EVAL_P]))
    r = {"score": float(np.mean(sc)), "neg": float(np.mean(np.array(sc) < 0)), "chars": float(np.mean([len(text(r)) for _, r in outs])),
         "distinct2": len(set(grams)) / max(len(grams), 1), "dup": 1 - len({tuple(r) for _, r in outs}) / len(outs),
         "kl": float((lp - ref).mean()), "pref_acc": float((margin > 0).float().mean())}
    r["qa"] = sum(any(k in reply(model, q) for k in keys) for q, keys in QA) / len(QA)
    if show:
        r["text"] = [text(rr) for _, rr in outs[:6]]
        r["other"] = [reply(model, o) for o in OTHER]
    return r

brief = lambda r: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k not in ("text", "other")}
lap = time.perf_counter()
before = evaluate(base, show=True)
print("학습 전:", brief(before), f"{time.perf_counter() - lap:.0f}초")
del base

def train_dpo(beta, epochs=3, lr=1e-4, batch=8, track=False, seed=0):
    """DPO 손실로 LoRA 행렬만 학습시킨다. 참조 모델의 로그 확률은 12.2절에서 미리 구한 값을 쓴다."""
    model = lora_model(seed)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    log = []
    for ep in range(1, epochs + 1):
        model.train(); order = random.Random(seed + ep).sample(range(len(TRAIN_P)), len(TRAIN_P))
        lap, st = time.perf_counter(), collections.defaultdict(list)
        for k in range(0, len(order), batch):
            qs = [TRAIN_P[i] for i in order[k:k + batch]]
            lp = seq_logp(model, [(q["prompt"], q["good"]) for q in qs] + [(q["prompt"], q["bad"]) for q in qs])
            lg, lb = lp[:len(qs)], lp[len(qs):]
            rg_, rb_ = torch.tensor([q["ref_good"] for q in qs]), torch.tensor([q["ref_bad"] for q in qs])
            loss, rg, rb = dpo_loss(lg, lb, rg_, rb_, beta)
            opt.zero_grad(); loss.mean().backward(); opt.step()
            st["loss"] += loss.tolist(); st["acc"] += (rg > rb).float().tolist()
            st["d_good"] += (lg - rg_).tolist(); st["d_bad"] += (lb - rb_).tolist()
        entry = {"epoch": ep, "sec": time.perf_counter() - lap, "loss": float(np.mean(st["loss"])), "train_acc": float(np.mean(st["acc"])),
                 "d_good": float(np.mean(st["d_good"])), "d_bad": float(np.mean(st["d_bad"]))}
        if track or ep == epochs:
            entry.update(evaluate(model, show=(ep == epochs)))
        log.append(entry)
        print(brief(entry), flush=True)
    return model, log

runs = {}
model, runs["β=0.1"] = train_dpo(0.1, track=True)
for t in runs["β=0.1"][-1]["text"]:
    print("  영화평 →", t[:70])
for o, t in zip(OTHER, runs["β=0.1"][-1]["other"]):
    print(f"  {o[:24].replace(chr(10), ' ')} → {t[:60]}")
del model

## 12.4 β 바꾸기

In [ ]:
for beta in [0.01, 1.0]:
    model, runs[f"β={beta}"] = train_dpo(beta)
    for t in runs[f"β={beta}"][-1]["text"][:4]:
        print("  영화평 →", t[:70])
    for o, t in zip(OTHER, runs[f"β={beta}"][-1]["other"]):
        print(f"  {o[:24].replace(chr(10), ' ')} → {t[:60]}")
    del model
print("학습 전 영화평:")
for t in before["text"][:4]:
    print("  영화평 →", t[:70])

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": True})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
R = runs["β=0.1"]; ep = [0] + [e["epoch"] for e in R]
fig, axes = plt.subplots(2, 2, figsize=(5.2, 5.0), dpi=260)
a = axes[0, 0]
a.plot(ep, [0] + [e["d_good"] for e in R], "-o", color=BLUE, ms=3.5, label="좋은 답")
a.plot(ep, [0] + [e["d_bad"] for e in R], "-o", color=RED, ms=3.5, label="나쁜 답")
a.axhline(0, color=GRAY, lw=0.8, ls=":")
a.set_title("① log π − log π_ref(에폭 평균)", loc="left", fontsize=8.6, color="#374151"); a.legend(frameon=False, fontsize=8)
a = axes[0, 1]
a.plot(ep, [before["score"]] + [e["score"] for e in R], "-o", color=PURPLE, ms=3.5, label="판정 점수")
a2 = a.twinx(); a2.plot(ep, [before["chars"]] + [e["chars"] for e in R], "-s", color=ORANGE, ms=3.5, label="글자 수")
a.set_title("② 검증 질문의 답(β=0.1)", loc="left", fontsize=8.6, color="#374151")
a.set_ylabel("판정 점수", color=PURPLE, fontsize=8); a2.set_ylabel("글자 수", color=ORANGE, fontsize=8)
order = sorted(runs, key=lambda k: float(k.split("=")[1]))          # β가 작은 것부터
names = ["학습 전"] + order
rows = [before] + [runs[k][-1] for k in order]
x = np.arange(len(names))
a = axes[1, 0]
a.bar(x - 0.2, [r["neg"] * 100 for r in rows], 0.38, color=BLUE, label="부정 쪽 비율(%)")
a.bar(x + 0.2, [r["dup"] * 100 for r in rows], 0.38, color=RED, label="같은 답 비율(%)")
a.set_xticks(x, names); a.set_title("③ 3에폭 뒤", loc="left", fontsize=8.6, color="#374151")
a.set_ylim(0, 105); a.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, handlelength=1.0, columnspacing=0.8)
a = axes[1, 1]
a.bar(x, [r["kl"] for r in rows], 0.55, color=GREEN)
for i, r in enumerate(rows):
    a.text(i, r["kl"] + 0.7, f"{r['kl']:.1f}", ha="center", va="bottom", fontsize=8)
a.set_xticks(x, names); a.set_title("④ 참조 모델과의 KL(나트)", loc="left", fontsize=8.6, color="#374151")
for a in axes.flat:
    a.spines[["top"]].set_visible(False)
for a in [axes[0, 0], axes[1, 0], axes[1, 1]]:
    a.spines[["right"]].set_visible(False)
for a in axes[0]:
    a.set_xlabel("에폭(0은 학습 전)", fontsize=8); a.set_xticks(range(0, len(ep)))
fig.tight_layout(); fig.savefig("fig_12_dpo.png"); plt.close(fig)